## Building a Chatbot

In this Notebook we will go over an example of how to design and implement a LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot will only use the language model to have a conversation. There are several other related concepts that we may be looking for:
- Conversational RAG: Enable a chatbot experience over an external source of data
- AI Agents: Build a chatbot that can execute an action



In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.environ['GROQ_API_KEY']

print(f'GROQ API KEY: {GROQ_API_KEY[:3]}**{GROQ_API_KEY[-3:]}')

GROQ API KEY: gsk**qNU


### Create a Chatbot from GROQ

Initialize **ChatGroq**. To initialize ChatGroq we need to pass the model name. All the model names can be found in [Groq docs](https://console.groq.com/docs/models).

In [6]:
from langchain_groq import ChatGroq

model = ChatGroq(model='Gemma2-9b-It', api_key=GROQ_API_KEY)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x12206d400>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12206dfd0>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

Simple Human message to the LLM

In [7]:
from langchain_core.messages import HumanMessage

response = model.invoke([HumanMessage(content='Hi, I am Dimo and I am Java developer.')])
response

AIMessage(content="Hi Dimo! It's nice to meet you. \n\nThat's great that you're a Java developer! It's a powerful and versatile language. \n\nIs there anything specific you'd like to talk about regarding Java? Perhaps you have a project you're working on, a question you need help with, or just want to discuss the latest trends in the Java world? I'm here to chat!\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 21, 'total_tokens': 113, 'completion_time': 0.167272727, 'prompt_time': 0.00132427, 'queue_time': 0.200446243, 'total_time': 0.168596997}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--57734a27-6042-4258-9d2a-b8471a6ec7a6-0', usage_metadata={'input_tokens': 21, 'output_tokens': 92, 'total_tokens': 113})

In [8]:
from langchain_core.messages import AIMessage

response = model.invoke([
    HumanMessage(content='Hi, I am Dimo and I am Java developer.'),
    AIMessage(content="Hi Dimo, \n\nIt's nice to meet you! I'm a large language model, here to help with any questions or tasks you might have. \n\nWhat can I do for you today? Are you working on a particular Java project you'd like to discuss, or do you have a coding problem you need help with? \n\nI'm always happy to chat about Java or any other programming language.\n"),
    HumanMessage(content='Do you remember my name and my profession?')
])
response

AIMessage(content='Yes, I do! 😊  My memory is pretty good.\n\nYou introduced yourself as Dimo, a Java developer.  \n\nIs there anything I can help you with related to your work today?\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 129, 'total_tokens': 173, 'completion_time': 0.08, 'prompt_time': 0.00337077, 'queue_time': 0.201203794, 'total_time': 0.08337077}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--1c817402-9b76-4c2f-8224-227a7e6e89e6-0', usage_metadata={'input_tokens': 129, 'output_tokens': 44, 'total_tokens': 173})

### Message History

How to store context in a session between LLM and Human. We can use **MessageHistory** class to *wrap* our model and make it **stateful**. This will keep track of inputs and outputs of the model and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. 


But when multiple users are chatting with the LLM, How we will differentiate one user from another. For that reason we will define function `get_session_history()`. This function will create a **SESSION_ID** and this will return `BaseChatMessageHistory`. 

In [9]:
# In memory implementation of chat message history. Stores messages in a memory list.
from langchain_community.chat_message_histories import ChatMessageHistory
# Abstract base class for storing chat message history.
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(runnable=model, get_session_history=get_session_history)

In [10]:
config = {'configurable': {'session_id': 'chat1'}}

In [11]:
response = with_message_history.invoke([
    HumanMessage(content='Hi, I am Dimo and I am Java developer.')
], config=config)
response

AIMessage(content="Hello Dimo!\n\nIt's nice to meet you.  I'm glad to know you're a Java developer. \n\nWhat kind of projects are you working on these days? Do you have any specific questions about Java or need help with something? I'm here to assist if I can!\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 21, 'total_tokens': 88, 'completion_time': 0.121818182, 'prompt_time': 0.00132368, 'queue_time': 0.199848176, 'total_time': 0.123141862}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--c42f508a-b527-4536-87f4-8412549aba48-0', usage_metadata={'input_tokens': 21, 'output_tokens': 67, 'total_tokens': 88})

In [13]:
response_2 = with_message_history.invoke([
    HumanMessage('Do you remember my name and profession?')
], config=config)
response_2

AIMessage(content="Yes, I do! I remember you're Dimo, a Java developer. 😊  \n\nIs there anything I can help you with regarding Java or your projects?  I'm ready when you are!\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 104, 'total_tokens': 150, 'completion_time': 0.083636364, 'prompt_time': 0.002749608, 'queue_time': 0.202809937, 'total_time': 0.086385972}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--b308c8fc-90f3-4a10-af4e-796fc6a53bd8-0', usage_metadata={'input_tokens': 104, 'output_tokens': 46, 'total_tokens': 150})

If we change the Session ID

In [14]:
config_2 = {'configurable': {'session_id': 'chat2'}}
response_3 = with_message_history.invoke([
    HumanMessage('Do you remember my name and profession?')
], config=config_2)
response_3

AIMessage(content="As a large language model, I have no memory of past conversations. Every interaction we have is a fresh start.\n\nIf you'd like to tell me your name and profession, I'd be happy to know! 😊\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 17, 'total_tokens': 66, 'completion_time': 0.089090909, 'prompt_time': 0.001251163, 'queue_time': 0.201985257, 'total_time': 0.090342072}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--85b2198f-f48e-4ae0-a2a8-afc63e8ebe6e-0', usage_metadata={'input_tokens': 17, 'output_tokens': 49, 'total_tokens': 66})

In [15]:
store

{'chat1': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi, I am Dimo and I am Java developer.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Dimo!\n\nIt's nice to meet you.  I'm glad to know you're a Java developer. \n\nWhat kind of projects are you working on these days? Do you have any specific questions about Java or need help with something? I'm here to assist if I can!\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 21, 'total_tokens': 88, 'completion_time': 0.121818182, 'prompt_time': 0.00132368, 'queue_time': 0.199848176, 'total_time': 0.123141862}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--c42f508a-b527-4536-87f4-8412549aba48-0', usage_metadata={'input_tokens': 21, 'output_tokens': 67, 'total_tokens': 88}), HumanMessage(content='Do you remember my name and profession?', addition

## Prompt templates

Prompt templates help to turn raw user information into a format that the LLM can work with. In this case the raw user input is just a message, which we are passing to the LLM. Let's now make that a little bit more complicated. First let's add a system message with some custom instructions (but still taking messages as input). Next we will add in more input besides just the messages.

In [16]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate(messages=[
    ('system', 'You are a helpful assistant. Answer all the questions to the best of your ability.'),
    MessagesPlaceholder(variable_name='messages')
])

chain = prompt | model 

In [17]:
chain.invoke(input={'messages': [HumanMessage(content='Hi, My name is Dimo')]})

AIMessage(content="Hello Dimo, it's nice to meet you! \n\nHow can I help you today?  😊  \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 33, 'total_tokens': 61, 'completion_time': 0.050909091, 'prompt_time': 0.00147828, 'queue_time': 0.201353237, 'total_time': 0.052387371}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--38f2af0d-4608-41dd-9737-665168f0f5ff-0', usage_metadata={'input_tokens': 33, 'output_tokens': 28, 'total_tokens': 61})

In [19]:
with_message_history = RunnableWithMessageHistory(runnable=chain, get_session_history=get_session_history)

config = {'configurable': {'session_id': 'newSession1'} }

response = with_message_history.invoke(input={'messages': [HumanMessage(content='Hi, My name is Dimo and I am a Java developer')]},
                                       config=config)
response

AIMessage(content="Hello Dimo, nice to meet you! 👋\n\nI'm glad you're here. How can I help you today?  \n\nAre you working on a specific project, or do you have a general Java question? I can help with things like:\n\n* **Explaining Java concepts:**  If you're stuck on something, I can try to explain it in a clear and concise way.\n* **Finding code examples:** I can search for relevant code snippets to help you get started.\n* **Debugging your code:**  While I can't directly execute your code, I can help you identify potential issues and suggest solutions.\n* **Learning about new Java features:** I can provide information about the latest Java updates and features.\n\n\nJust let me know what you need! 😊\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 39, 'total_tokens': 205, 'completion_time': 0.301818182, 'prompt_time': 0.001563421, 'queue_time': 0.201680059, 'total_time': 0.303381603}, 'model_name': 'Gemma2-9b-It', 'system_finge

In [20]:
response = with_message_history.invoke(input={'messages': [HumanMessage(content='What do you think about my profession?')]},
                                       config=config)
response

AIMessage(content="I think being a Java developer is awesome! \n\nJava is a powerful and versatile language used in so many different areas, from web development to enterprise applications to Android apps.  It's a great language to learn and build a career around. \n\nWhat do you enjoy most about being a Java developer? \n\nIs there anything you find particularly challenging?\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 221, 'total_tokens': 297, 'completion_time': 0.138181818, 'prompt_time': 0.005468371, 'queue_time': 0.204093586, 'total_time': 0.143650189}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--235582e1-9d80-4d2a-babb-b15823a17c1d-0', usage_metadata={'input_tokens': 221, 'output_tokens': 76, 'total_tokens': 297})

Let's try this with **multiple variables**

In [21]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate(messages=[
    ('system', 'You are a helpful assistant. Answer all the questions in {language} to the best of your ability.'),
    MessagesPlaceholder(variable_name='messages')
])

chain = prompt | model 

response = chain.invoke(input={ 'messages': [HumanMessage('Hi, I am Dimo and I am Java developer')], 'language': 'Bulgarian' })
response

AIMessage(content='Здравей, Димо! Радвам се да те срещна.  Като Java разработчик, сигурно имаш много интересни проекти. Как мога да ти помогна днес? 😊  \n\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 39, 'total_tokens': 88, 'completion_time': 0.089090909, 'prompt_time': 0.00155433, 'queue_time': 0.199783443, 'total_time': 0.090645239}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--2db4dca1-b3bb-48be-b895-7318cd78645f-0', usage_metadata={'input_tokens': 39, 'output_tokens': 49, 'total_tokens': 88})

Let's wrap that more complex chain in a message history class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history. 

In [22]:
with_message_history = RunnableWithMessageHistory(runnable=chain, get_session_history=get_session_history,
                                                  input_messages_key='messages')

In [23]:
config = {'configurable': {'session_id': 'chat4'}}
response = with_message_history.invoke(input={
    'messages': [HumanMessage(content='Hi I am Dimo and I am Java developer')],
    'language': 'Bulgarian'
}, config=config)
response

AIMessage(content='Здравей, Димо! Е хубаво да те срещна. \n\nКазваш се Димо и си Java разработчик? Това е страхотно!  Какво мога да направя за теб днес?\n\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 38, 'total_tokens': 90, 'completion_time': 0.094545455, 'prompt_time': 0.001555788, 'queue_time': 0.203468676, 'total_time': 0.096101243}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--5cef20de-742a-4092-b254-95fe79009193-0', usage_metadata={'input_tokens': 38, 'output_tokens': 52, 'total_tokens': 90})

In [24]:
store['chat4']

InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi I am Dimo and I am Java developer', additional_kwargs={}, response_metadata={}), AIMessage(content='Здравей, Димо! Е хубаво да те срещна. \n\nКазваш се Димо и си Java разработчик? Това е страхотно!  Какво мога да направя за теб днес?\n\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 38, 'total_tokens': 90, 'completion_time': 0.094545455, 'prompt_time': 0.001555788, 'queue_time': 0.203468676, 'total_time': 0.096101243}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--5cef20de-742a-4092-b254-95fe79009193-0', usage_metadata={'input_tokens': 38, 'output_tokens': 52, 'total_tokens': 90})])

In [25]:
response = with_message_history.invoke(input={
    'messages': [HumanMessage(content='What do you think about my profession')],
    'language': 'Bulgarian'
}, config=config)
response

AIMessage(content='Java е невероятно популярна езикова програмиране, и като Java разработчик си в много добра позиция!  Има голям брой възможности в тази област, и уменията ти са високо ценени.  \n\nКакво те харесва най-много в професията ти? \n\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 104, 'total_tokens': 173, 'completion_time': 0.125454545, 'prompt_time': 0.002818168, 'queue_time': 0.198784318, 'total_time': 0.128272713}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--6ba13c83-5eb7-44b2-ac7f-0118e526988a-0', usage_metadata={'input_tokens': 104, 'output_tokens': 69, 'total_tokens': 173})

### Manage the Conversation History

An important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is **Important** to add a step that limits the size of messages you are passing in.

For that purpose we will use `trim_messages` method from `langchain_core.messages`. What exactily `trim_messages` method does? This is a helper which will reduce how many messages we will send to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to keep the system message and whether to allow partial messages.

In [35]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=60, strategy='last',
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on='human'
)

messages = [
    SystemMessage(content='You are a good assistant'),
    HumanMessage(content='Hi, I am Bob'),
    AIMessage(content='Hi, Bob!'),
    HumanMessage(content='What is 2 + 2?'),
    AIMessage(content='2 + 2 is equal to 4'),
    HumanMessage(content='Thanks'),
    AIMessage(content='No problem'),
    HumanMessage(content='I like chocolate ice cream'),
    AIMessage(content='nice'),
    HumanMessage(content='Are you having fun'),
    AIMessage(content='Oh yes, of cource')
]
trimmer.invoke(messages)

[SystemMessage(content='You are a good assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is 2 + 2?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='2 + 2 is equal to 4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='No problem', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like chocolate ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Are you having fun', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Oh yes, of cource', additional_kwargs={}, response_metadata={})]

We can also pass the trimmer in a **chain**

In [36]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (RunnablePassthrough.assign(messages=itemgetter('messages') | trimmer)) | prompt | model

response = chain.invoke(input={
    'messages': messages + [HumanMessage(content='What Ice cream do I like?')],
    'language': "Bulgarian"
})

response

AIMessage(content='Според това, което каза, ти харесва шоколадово сладолед. \n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 90, 'total_tokens': 112, 'completion_time': 0.04, 'prompt_time': 0.002572778, 'queue_time': 0.196783848, 'total_time': 0.042572778}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--52a37c59-c716-434c-bff5-0d3c61f66e92-0', usage_metadata={'input_tokens': 90, 'output_tokens': 22, 'total_tokens': 112})

Let's wrap it in **RunnableWithMessageHistory**

In [37]:
with_message_history = RunnableWithMessageHistory(
    runnable=chain,
    get_session_history=get_session_history,
    input_messages_key='messages'
)

config = {
    'configurable': {'session_id': 'chat5'}
}

In [40]:
response = with_message_history.invoke(input={
    'messages': messages + [HumanMessage('What is my name')],
    'language': 'Bulgarian'
}, config=config)

response

AIMessage(content="I don't know your name. I don't have access to past conversations or personal information about you.  \n\nIf you'd like to tell me your name, I'd be happy to know! 😊 \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 87, 'total_tokens': 137, 'completion_time': 0.090909091, 'prompt_time': 0.002503078, 'queue_time': 0.192722453, 'total_time': 0.093412169}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--479f5be5-f689-4ed1-9add-4694467286ec-0', usage_metadata={'input_tokens': 87, 'output_tokens': 50, 'total_tokens': 137})